# Final Out-of-Sample Evaluation

## Objective

Evaluate the selected Tuned Random Forest against two simple
baselines using previously untouched test data.

Our model was selected using validation RMSE in notebook 03.

Selected Random Forest hyperparameters:
- n_estimators = 200
- max_depth = 3
- min_samples_leaf = 10
- random_state = 42

Primary evaluation metric: RMSE
Secondary evaluation metric: MAE

No further model tuning or selection will be performed using
test results.

In [19]:
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

In [20]:
# Locate the project directory
project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

# Load our previously engineered dataset
csv_path = project_dir / "data" / "qqq_model_data.csv"

model_data = pd.read_csv(
    csv_path,
    index_col="Date",
    parse_dates=["Date"]
)

# Verify the loaded dataset
print("Total observations:", len(model_data))
print("First date:", model_data.index.min().date())
print("Last date:", model_data.index.max().date())

model_data.head()

Total observations: 4170
First date: 2010-02-02
Last date: 2026-08-31


,Daily_Return_Pct,Momentum_5d_Pct,Volatility_20d_Pct,MA_Distance_20d_Pct,Target_Next_Day_Return_Pct
Date,,,,,
2010-02-02,0.901552,-1.578332,1.265401,-3.512449,0.549851
2010-02-03,0.549851,-1.812001,1.278160,-2.709864,-2.893640
2010-02-04,-2.893640,-2.135501,1.405649,-5.155065,0.844653
2010-02-05,0.844653,0.444014,1.429440,-4.013255,-0.721268
2010-02-08,-0.721268,-1.363864,1.404370,-4.290909,1.031256


## 1. Prepare the Final Evaluation Data

Recreate the original chronological split, preserving the same
training, validation, and testing periods used during model development.

In [21]:
# Define our four predictive features
feature_cols = [
    "Daily_Return_Pct",
    "Momentum_5d_Pct",
    "Volatility_20d_Pct",
    "MA_Distance_20d_Pct"
]

target_col = "Target_Next_Day_Return_Pct"

# Recreate the original chronological split
n = len(model_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_data = model_data.iloc[:train_end - 1].copy()
val_data = model_data.iloc[train_end:val_end - 1].copy()
test_data = model_data.iloc[val_end:].copy()

# Verify the dataset sizes and dates
for name, data in [
    ("Training", train_data),
    ("Validation", val_data),
    ("Testing", test_data)
]:
    print(name)
    print("Rows:", len(data))
    print("Dates:", data.index.min().date(), "to", data.index.max().date())
    print()

Training
Rows: 2918
Dates: 2010-02-02 to 2021-09-02

Validation
Rows: 624
Dates: 2021-09-07 to 2024-02-29

Testing
Rows: 626
Dates: 2024-03-04 to 2026-08-31



## 2. Reconstruct the Selected Random Forest

Train the previously selected Random Forest using its frozen
hyperparameters.

No additional tuning or model selection is performed.

In [22]:
# Separate predictive features and targets
X_train = train_data[feature_cols]
y_train = train_data[target_col]

X_test = test_data[feature_cols]
y_test = test_data[target_col]

# Reconstruct our selected Random Forest
final_rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=3,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

# Train using the original training observations
final_rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",3
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max

## 3. Generate Out-of-Sample Predictions

Generate predictions on the previously untouched test set
using the selected Random Forest and two simple baselines.

In [23]:
# Calculate the training mean return
training_mean = y_train.mean()

# Generate predictions from each approach
test_predictions_by_model = {
    "Predict 0%": np.zeros(len(y_test)),
    "Predict training mean": np.full(len(y_test), training_mean),
    "Tuned Random Forest": final_rf_model.predict(X_test)
}

print("Number of test observations:", len(y_test))
print("Training mean return:", training_mean)

Number of test observations: 626
Training mean return: 0.08582604660578723


## 4. Final Out-of-Sample Performance

Compare test performance using RMSE as the primary evaluation
metric and MAE as a secondary metric.

These results are reported without further model tuning.

In [24]:
# Calculate test performance for each approach
test_performance_rows = []

for model_name, predictions in test_predictions_by_model.items():

    test_performance_rows.append({
        "Model": model_name,
        "Test MAE": mean_absolute_error(y_test, predictions),
        "Test RMSE": root_mean_squared_error(y_test, predictions)
    })

# Create and sort the comparison table
test_performance = pd.DataFrame(test_performance_rows)

test_performance = test_performance.sort_values(
    by="Test RMSE",
    ascending=True
).reset_index(drop=True)

test_performance.round(6)

,Model,Test MAE,Test RMSE
0,Tuned Random Forest,0.934910,1.314285
1,Predict training mean,0.937274,1.340725
2,Predict 0%,0.945665,1.343452


In [25]:
rf_test_preds = test_predictions_by_model["Tuned Random Forest"]

print("RANDOM FOREST PREDICTION ANALYSIS")

print("\nPredicted returns:")
print("Mean:", np.mean(rf_test_preds))
print("Standard deviation:", np.std(rf_test_preds))

print("\nActual returns:")
print("Mean:", y_test.mean())
print("Standard deviation:", y_test.std())

print("\nPrediction-actual correlation:")
print(np.corrcoef(rf_test_preds, y_test)[0, 1])

RANDOM FOREST PREDICTION ANALYSIS

Predicted returns:
Mean: 0.10163449138647959
Standard deviation: 0.2055901204177546

Actual returns:
Mean: 0.08555518426051638
Standard deviation: 1.3417973114884152

Prediction-actual correlation:
0.20447708091351977


## 5. Feature Importance

Examine which predictive features the Random Forest relied
on most when constructing its decision trees.

In [26]:
# Extract feature importance scores
feature_importance = pd.Series(
    final_rf_model.feature_importances_,
    index=feature_cols
)

# Sort features from most to least important
feature_importance = feature_importance.sort_values(
    ascending=False
)

feature_importance

MA_Distance_20d_Pct    0.410694
Momentum_5d_Pct        0.211714
Daily_Return_Pct       0.205710
Volatility_20d_Pct     0.171882
dtype: float64

## 6. Final Findings and Limitations

### Out-of-Sample Performance

The selected Random Forest outperformed both constant-return
baselines on the untouched 2023–2026 test period.

Test RMSE:
- Tuned Random Forest: 1.314285
- Training-mean baseline: 1.340725
- Zero-return baseline: 1.343452

The Random Forest reduced test RMSE by approximately 2.17%
relative to predicting zero returns.

### Prediction Characteristics

- Prediction-actual correlation: 0.2045
- Mean predicted return: 0.1016%
- Mean actual return: 0.0856%
- Predicted return standard deviation: 0.2056
- Actual return standard deviation: 1.3418

The model's predictions were substantially less volatile
than actual returns, indicating that it produced relatively
conservative forecasts.

### Feature Importance

The most influential feature was distance from the
20-day moving average (41.1%), followed by:

- 5-day momentum: 21.2%
- Daily return: 20.6%
- 20-day volatility: 17.2%

These scores describe the fitted model's use of features,
not causal relationships.

### Limitations

- The analysis covers only QQQ.
- Evaluation uses one final historical test period.
- Predictive improvement is modest.
- Statistical significance has not been established.
- Lower forecasting error does not guarantee profitability.
- Transaction costs and realistic trade execution have not
  yet been evaluated.

### Conclusion

The tuned Random Forest demonstrated modest out-of-sample
predictive improvement over simple baselines.

The results provide preliminary evidence of a weak
forecasting signal, but further research is required
before concluding that it has practical trading value.